**Version 3.0 (CCE in a cross-section, FDI inward)** of `estimation_all_bases_cce_2.0.ipynb`: for the bases estimated on ONE year (`ONLY_YEAR`, FDI), the CCE needs a time dimension that does not exist. This version uses the **sectors as the periods**: the units are the countries, the cross-section averages are taken by sector over the countries, and each country is purged of (CCEP) or regressed on (CCEMG) these averages. For these bases the columns `CCEP` and `CCEMG` hold this version (the column `effects` of the sheet `raw` says: countries = units, sectors = periods). They assume that the coefficients are the same for all the sectors of a country and that the common factors are sector-specific, and they need a stock that varies across the sectors of a country: this is the case of the FDI **inward** stock, not of the **outward** stock of the panel `panel_FDI_outwards_F.csv`, which is the same for all the sectors of a country (the CCE is then skipped).

**Version 2.0 (FDI inward and outward)** of `estimation_all_bases_cce.ipynb`: the FDI base is estimated for the two measures, **inward** and **outward** (option `FDI_VARIANTS`), from the panels of `RESULTS_DIR/regression_panels/` (see `load_fdi`: `panel_FDI_F.csv` of `panel_setup_2.0.ipynb`, or one panel per measure such as `panel_FDI_outwards_F.csv`). Each (base, metric) now has its own output file, for instance `tab_FDI_dir_inward_cce.xlsx` and `tab_FDI_dir_outward_cce.xlsx` (the metric of the estimates is then `dir_inward` / `dir_outward`).

# Estimations — country fixed effects, year fixed effects and Common Correlated Effects (CCE)

Estimation of the unit-cost equation

    ln(c_u / w_l) = alpha * ln(C_k / w_l) + gamma' X + e,        X = ln knowledge stocks

for the formulations of `estimation_all_bases.ipynb`, with four methods (note "Econometric strategy"), by sector and on pooled groups of sectors. The code follows `estimations_all.py` (Estim_TFP).

**Methods**

| Method | What it does |
| --- | --- |
| Country FE | country fixed effects (country x sector in the pooled estimations) + a dummy for 2009 |
| Country FE + year FE | the same with year fixed effects (the 2009 dummy is then absorbed) |
| CCEP | CCE pooled (Pesaran 2006): every unit is purged of a constant and of the cross-section averages (by sector and year) of y and of every regressor; the residuals of all the units are pooled and one coefficient is estimated |
| CCEMG | CCE mean group: one regression per unit with the same averages; mean of the coefficients of the units (5 % trimmed on each side), standard error = standard deviation / square root of the number of units |

Standard errors of Country FE, Country FE + year FE and CCEP: clustered by country.

**Formulations** (the stocks are in logs): (1) total; (2) national / foreign; (3) intra / inter-sectoral; (4) four channels; (5A) stock only, (5B) stock + own R&D intensity (R/Y in %), (5C) stock + intensity + their product (absorptive capacity), the three on the same sample.

**Samples**: `by_sector` = each sector separately (units = countries); pooled groups (units = country x sector, the coefficient of the relative price of capital specific to each sector): manufacturing, transport, ICT, all sectors, market sectors (the lists are in the options cell). The aggregates "C" and "H" are left out of the groups to avoid double counting. `OECD_ICIO` (2016-2022 only) is estimated on the pooled groups with the two fixed-effects methods only.

- `ONLY_YEAR` (options cell): for the bases listed there (FDI: 2017) the panel keeps every year but the estimation uses a single year. With one year there is no time dimension, so unit effects, year effects and the CCE cannot be computed: only the pooled groups are estimated, by OLS with country effects, sector intercepts and alpha specific to each sector (shown in the column "Country FE").
- Input: `RESULTS_DIR/regression_panels/panel_{BASE}_F.csv`, produced by `panel_setup` (several bases can be estimated in one run: option `RUNS`).
- Output: one Excel file and one csv per (base, metric) in `RESULTS_DIR/estimation_results/<BASE>/` (suffix `_cce`; the reference tables are not overwritten). Cells: coefficient with stars (*** 1 %, ** 5 %, * 10 %). A negative coefficient means that the knowledge stock lowers the unit cost, i.e. raises TFP.

**Run All** estimates both direct and embodied knowledge for the configured base when supported, plus embodied investment for CAPITAL. Edit RUNS in the options cell to select other base/metric pairs. Each pair writes an Excel workbook and a CSV.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from linearmodels.panel import PanelOLS
import importlib
import sys
from pathlib import Path

for parent in (Path.cwd(), *Path.cwd().parents):
    code_dir = parent if (parent / "project_settings.py").is_file() else parent / "code"
    if (code_dir / "project_settings.py").is_file():
        sys.path.insert(0, str(code_dir))
        break
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_SPE, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, CHOC_TAG
warnings.filterwarnings("ignore")
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")
print("RESULTS_DIR exists:", os.path.isdir(RESULTS_DIR), "| PATH_WRK =", PATH_WRK)

In [ ]:
# ====== What is estimated ======
RUNS = [(BASE, metric) for metric in ("dir", "embod", "embod_investment") if f"{BASE}_{metric}" in METRIC_SPE]
# Override RUNS to select specific pairs, e.g. [("EXIOBASE", "dir"), ("FIGARO", "embod")].
ONLY_POOLED = False              # True: only the pooled groups of sectors (much faster, no sector-by-sector estimation)
YEAR_FE_BY_SECTOR = False        # Country FE + year FE in the pooled groups: year effects common to all the sectors of the group (False, as in estimations_all.py)
                                 # or specific to each sector (True)
NO_CCE_BASES = ["OECD_ICIO"]     # bases estimated on the pooled groups with the two fixed-effects methods only (series too short for the CCE)
OUT_TAG = "_cce"                 # suffix of the exported file, so that the reference tables are not overwritten
FDI_VARIANTS = ["inward", "outward"]   # FDI measures estimated (panel columns rd_dir_inward*, rd_dir_outward*, built by panel_setup_2.0)
ONLY_YEAR = {"FDI": 2017}        # base -> year kept for the estimation (the panel keeps every year, only the estimation sample is restricted).
                                 # One year = a cross-section: see cross_section() below for what is estimated in that case

# ====== Samples ======
MANUF = ["C10_C12", "C13_C15", "C16_C18", "C19", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
TRANSPORT = ["H49", "H50", "H51", "H52"]
ICT = ["H53+J", "I"]            # as requested (NB: in NACE Rev.2, I is accommodation and food services)
OTHER = ["A", "B", "D", "E", "F", "G", "K", "L", "M+N", "O+U", "P", "Q", "R+S+T"]
ALL = MANUF + TRANSPORT + ICT + OTHER      # every elementary sector; the aggregates C and H are left out to avoid double counting
MARKET = [s for s in ALL if s not in ("L", "O+U", "P", "Q")]   # without real estate (imputed rents) and the mostly non-market sectors (output measured by inputs)
GROUPS = {"manufacturing": MANUF, "transport": TRANSPORT, "ICT": ICT, "all sectors": ALL, "market sectors": MARKET}

# ====== Formulations (as in estimation_all_bases.ipynb) ======
BLOCKS = {"intrasectoral_national_know": "rd_same_sector_same_country",
          "intrasectoral_foreign_know": "rd_same_sector_diff_country",
          "intersectoral_national_know": "rd_diff_sector_same_country",
          "intersectoral_foreign_know": "rd_diff_sector_diff_country"}
FORMULATIONS = {"(1) total": ["total_know"],
                "(2) national/foreign": ["national_know", "foreign_know"],
                "(3) intra/inter-sectoral": ["intrasectoral_know", "intersectoral_know"],
                "(4) four channels": list(BLOCKS),
                "(5A) stock only": ["total_know"],
                "(5B) stock + intensity": ["total_know", "own_rd_intensity"],
                "(5C) stock + intensity + interaction": ["total_know", "own_rd_intensity", "total_know_x_intensity"]}
# The three cases of model (5) are estimated on the same sample (the one of case C, where the own R&D intensity is known):
SAMPLE_VARS = {"(5A) stock only": ["total_know", "own_rd_intensity"],
               "(5B) stock + intensity": ["total_know", "own_rd_intensity"]}
METHODS = ["Country FE", "Country FE + year FE", "CCEP", "CCEMG"]
MIN_OBS_UNIT = 1          # units with fewer observations are dropped
CCE_SECTOR_DIM = True    # one-year bases (ONLY_YEAR): also estimate the CCE with the countries as units and the sectors as periods
MIN_SECTORS_CCE = 8      # CCE over sectors: minimum number of sectors per country
MIN_COUNTRIES_SECTOR = 5 # CCE over sectors: a sector needs at least this many countries (its cross-section average is otherwise degenerate)
MIN_DOF_MG = 3           # CCEMG: a unit regression is run only if it has at least this many degrees of freedom (observations - regressors)
MIN_T_CCE = 1            # CCE: minimum observations per unit


def panel_file(base):
    """Final full-sample panel of a base, produced by panel_setup."""
    return RESULTS_DIR + f"regression_panels/panel_{base}_F.csv"

In [ ]:
def stars(p):
    return "" if not np.isfinite(p) else "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""


FDI_SUFFIX = {"outward": "rd_dir_fdips", "inward": "rd_dir_fdips_inwards"}   # suffix of the knowledge columns of each FDI measure


def load_fdi(variant):
    """Panel and column names of one FDI measure ("inward" / "outward"), looked up in RESULTS_DIR/regression_panels/ in this order:
    1. panel_FDI_F.csv with the columns rd_dir_<variant>* (panel_setup_2.0.ipynb: both measures in one panel);
    2. panel_FDI_<variant>s_F.csv (one panel per measure, e.g. panel_FDI_outwards_F.csv, panel_FDI_inwards_F.csv): the total stock rd_<variant>s only;
    3. panel_FDI_F.csv of the previous panel_setup, which holds ONE measure (plain rd_dir* columns): inward if its channels end with _inwards, else outward.
    Returns (panel, names, file) where names gives the column of the total stock and of D, F, I, X (None = not in the panel) and the suffix of the 4 channels."""
    folder = RESULTS_DIR + "regression_panels/"
    p_all = folder + "panel_FDI_F.csv"
    d_all = pd.read_csv(p_all) if os.path.exists(p_all) else None
    if d_all is not None and f"rd_dir_{variant}" in d_all.columns:
        stem = f"rd_dir_{variant}"
        return d_all, dict(total=stem, D=stem + "_D", F=stem + "_F", I=stem + "_I", X=stem + "_X", suffix=FDI_SUFFIX[variant]), p_all
    p_one = folder + f"panel_FDI_{variant}s_F.csv"
    if os.path.exists(p_one):
        return pd.read_csv(p_one), dict(total=f"rd_{variant}s", D=None, F=None, I=None, X=None, suffix=None), p_one
    if d_all is not None and f"rd_same_sector_same_country_{FDI_SUFFIX[variant]}" in d_all.columns:
        return d_all, dict(total="rd_dir", D="rd_dir_D", F="rd_dir_F", I="rd_dir_I", X="rd_dir_X", suffix=FDI_SUFFIX[variant]), p_all
    raise FileNotFoundError(f"no panel of the FDI measure '{variant}' in {folder} (panel_FDI_F.csv with rd_dir_{variant}*, or panel_FDI_{variant}s_F.csv)")


def prepare(base, metric, variant=None):
    """Panel of a base with y = ln(c_u) - ln(w_l), k = ln(C_k) - ln(w_l) and the knowledge stocks of the formulations (logs, positive values only).
    variant (FDI only): "inward" or "outward". A stock that is not in the panel is left empty and the formulations that need it are skipped."""
    if variant:
        d, nm, src_file = load_fdi(variant)
        print(f"[FDI {variant}] {src_file}: total stock = {nm['total']}")
    else:
        d = pd.read_csv(panel_file(base))
        cands = [c for c in d.columns if c.startswith(f"rd_same_sector_same_country_rd_{metric}_")]
        if metric == "embod":
            cands = [c for c in cands if "investment" not in c]
        if not cands or f"rd_{metric}" not in d.columns:
            raise ValueError(f"{panel_file(base)}: missing knowledge columns for {metric}. Rebuild the panel with this metric available.")
        nm = dict(total=f"rd_{metric}", D=f"rd_{metric}_D", F=f"rd_{metric}_F", I=f"rd_{metric}_I", X=f"rd_{metric}_X",
                  suffix=cands[0][len("rd_same_sector_same_country_"):])
    if base in ONLY_YEAR:
        d = d[d["annee"] == ONLY_YEAR[base]].copy()
        print(f"[{base}] estimation sample restricted to {ONLY_YEAR[base]}: {len(d)} rows")
        if d.empty:
            raise RuntimeError(f"{base}: no row for the year {ONLY_YEAR[base]} in the panel")
    absent = []

    def lg(c):
        if c is None or c not in d.columns:
            absent.append(c)
            return pd.Series(np.nan, index=d.index)
        return np.log(d[c].where(d[c] > 0))

    with np.errstate(divide="ignore", invalid="ignore"):
        d["y"] = np.log(d["c_u"]) - np.log(d["w_l"])
        d["k"] = np.log(d["C_k"]) - np.log(d["w_l"])
        d["total_know"] = lg(nm["total"])
        d["national_know"], d["foreign_know"] = lg(nm["D"]), lg(nm["F"])
        d["intrasectoral_know"], d["intersectoral_know"] = lg(nm["I"]), lg(nm["X"])
        for name, pre in BLOCKS.items():
            d[name] = lg(f"{pre}_{nm['suffix']}" if nm["suffix"] else None)
        d["own_rd_intensity"] = d["int_propre"]
        d["total_know_x_intensity"] = d["own_rd_intensity"] * d["total_know"]
    if absent:
        print(f"[{base}{' ' + variant if variant else ''}] not in the panel (formulations that need them are skipped): {sorted(set(map(str, absent)))}")
    return d.replace([np.inf, -np.inf], np.nan)


def cross_section(base):
    """True if the estimation sample of the base is a single year."""
    return base in ONLY_YEAR


def sample(d, sectors, xs, min_obs=None):
    """Complete cases of the sectors, one unit = country x sector, units with at least min_obs (default MIN_OBS_UNIT) years."""
    g = d[d["Sector"].isin(sectors)].dropna(subset=["y", "k", "2009_crisis"] + xs).copy()
    g["unit"] = g["Region"] + "|" + g["Sector"]
    g = g[g.groupby("unit")["annee"].transform("size") >= (MIN_OBS_UNIT if min_obs is None else min_obs)]
    return g


def fe(g, xs, year_fe):
    """Fixed effects of the units (+ year effects if year_fe, else the 2009 dummy), standard errors clustered by country.
    Several sectors: the coefficient of k is specific to each sector (alpha_s)."""
    gi = g.set_index(["unit", "annee"]).sort_index()
    X = gi[xs].copy()
    secs = gi["Sector"].unique()
    if len(secs) > 1:
        for s in secs:
            X[f"k_{s}"] = np.where(gi["Sector"] == s, gi["k"], 0.0)
    else:
        X["k"] = gi["k"]
    if not year_fe:
        X["crisis"] = gi["2009_crisis"]
    X["const"] = 1.0
    X = X.loc[:, X.nunique() > 1].assign(const=1.0)
    by_sector = year_fe and YEAR_FE_BY_SECTOR and len(secs) > 1
    other = pd.DataFrame({"sector_year": pd.Categorical(gi["Sector"] + "_" + gi.index.get_level_values(1).astype(str))}, index=gi.index) if by_sector else None
    res = PanelOLS(gi["y"], X, entity_effects=True, time_effects=year_fe and not by_sector, other_effects=other, drop_absorbed=True, check_rank=False) \
        .fit(cov_type="clustered", clusters=pd.DataFrame({"c": pd.Categorical(gi["Region"]).codes}, index=gi.index))
    return {x: (res.params.get(x, np.nan), res.std_errors.get(x, np.nan), res.pvalues.get(x, np.nan)) for x in xs}, \
        int(res.nobs), gi.index.get_level_values(0).nunique()


def fe_cross(g, xs):
    """Cross-section (one year): country effects + sector intercepts (+ alpha_s specific to each sector if several sectors), OLS, standard errors
    clustered by country. There is no time dimension: no unit effects, no year effects, no CCE.
    If a stock is the same for all the sectors of a country (the FDI outward stock of a country), it is collinear with the country effects:
    the country effects are then dropped and only the sector intercepts are kept (effects = "sector"). Returns the effects used as 4th value."""
    g = g.reset_index(drop=True)
    cols = {x: g[x].to_numpy() for x in xs}
    secs = sorted(g["Sector"].unique())
    if len(secs) > 1:
        for s in secs:
            cols[f"k_{s}"] = np.where(g["Sector"] == s, g["k"], 0.0)
    else:
        cols["k"] = g["k"].to_numpy()
    X = pd.DataFrame(cols)
    within = lambda x: (g[x] - g.groupby("Region")[x].transform("mean")).abs().max() > 1e-8 * max(g[x].abs().max(), 1e-12)
    country_fe = all(within(x) for x in xs)
    if country_fe:
        X = pd.concat([X, pd.get_dummies(g["Region"], prefix="cty", dtype=float)], axis=1)
        if len(secs) > 1:
            X = pd.concat([X, pd.get_dummies(g["Sector"], prefix="sec", dtype=float, drop_first=True)], axis=1)
    elif len(secs) > 1:
        X = pd.concat([X, pd.get_dummies(g["Sector"], prefix="sec", dtype=float)], axis=1)
    else:
        X["const"] = 1.0
    b, se, G = cluster_ols(g["y"].to_numpy(), X.to_numpy(float), g["Region"].to_numpy())
    out = {}
    for x in xs:
        j = X.columns.get_loc(x)
        out[x] = (b[j], se[j], 2 * (1 - stats.t.cdf(abs(b[j] / se[j]), G - 1)) if se[j] > 0 else np.nan)
    return out, len(g), g["unit"].nunique(), "country + sector" if country_fe else "sector"


def cce_sector(g, xs):
    """CCE in a cross-section: the COUNTRIES are the units and their SECTORS play the role of the periods. The common factors are proxied by the
    cross-section averages, by sector and over the countries, of y, k and of every stock. CCEP: every country is purged of a constant and of these
    averages, the residuals of all the countries are pooled (alpha specific to each sector, standard errors clustered by country).
    CCEMG: one regression per country with the same averages, 5 % trimmed mean of the coefficients.
    Needs stocks that vary across the sectors of a country (otherwise they cannot be told apart from the constant): returns None if not,
    or if there are too few countries or sectors."""
    vars_ = ["y", "k"] + xs
    g = g.copy()
    if not all(((g[x] - g.groupby("Region")[x].transform("mean")).abs().max() > 1e-8 * max(g[x].abs().max(), 1e-12)) for x in xs):
        return "constant"
    for _ in range(2):    # sectors with enough countries, then countries with enough sectors (twice: each filter changes the other)
        g = g[g.groupby("Sector")["Region"].transform("nunique") >= MIN_COUNTRIES_SECTOR]
        g = g[g.groupby("Region")["Sector"].transform("nunique") >= MIN_SECTORS_CCE]
    if g["Region"].nunique() < 5 or g["Sector"].nunique() < 5:
        return None
    avg = g.groupby("Sector")[vars_].transform("mean")
    bars = []
    for v in vars_:
        g[f"bar_{v}"] = avg[v]
        bars.append(f"bar_{v}")
    parts, mg = [], []
    for u, gu in g.groupby("Region"):
        H = np.column_stack([np.ones(len(gu))] + [gu[b].to_numpy() for b in bars])
        P = np.linalg.pinv(H)
        R = {v: gu[v].to_numpy() - H @ (P @ gu[v].to_numpy()) for v in vars_}
        parts.append(pd.DataFrame({**R, "Sector": gu["Sector"].to_numpy(), "Region": gu["Region"].to_numpy()}))
        X = np.column_stack([np.ones(len(gu)), gu["k"].to_numpy()] + [gu[x].to_numpy() for x in xs] + [gu[b].to_numpy() for b in bars])
        if len(gu) >= X.shape[1] + MIN_DOF_MG:
            mg.append(np.linalg.lstsq(X, gu["y"].to_numpy(), rcond=None)[0][2:2 + len(xs)])
    R = pd.concat(parts, ignore_index=True)
    secs = sorted(R["Sector"].unique())
    kcols = [np.where(R["Sector"] == s, R["k"], 0.0) for s in secs] if len(secs) > 1 else [R["k"].to_numpy()]
    X = np.column_stack([R[x].to_numpy() for x in xs] + kcols)
    b, se, G = cluster_ols(R["y"].to_numpy(), X, R["Region"].to_numpy())
    mg = np.array(mg) if mg else np.empty((0, len(xs)))
    out = {}
    for j, x in enumerate(xs):
        p = 2 * (1 - stats.t.cdf(abs(b[j] / se[j]), G - 1)) if se[j] > 0 else np.nan
        res = {"CCEP": (b[j], se[j], p)}
        c = mg[:, j][np.isfinite(mg[:, j])] if len(mg) else np.array([])
        if len(c) >= 5:
            lo, hi = np.quantile(c, [0.05, 0.95])
            ct = c[(c >= lo) & (c <= hi)]
            s_ = ct.std(ddof=1) / np.sqrt(len(ct))
            res["CCEMG"] = (ct.mean(), s_, 2 * (1 - stats.norm.cdf(abs(ct.mean() / s_))))
        else:
            res["CCEMG"] = (np.nan, np.nan, np.nan)
        out[x] = res
    return out, len(g), g["Region"].nunique(), len(mg)


def cluster_ols(y, X, groups):
    """OLS with standard errors clustered by `groups` (small-sample factors G / (G - 1) and (n - 1) / (n - k))."""
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    meat = np.zeros((X.shape[1], X.shape[1]))
    for gr in np.unique(groups):
        m = groups == gr
        sg = X[m].T @ e[m]
        meat += np.outer(sg, sg)
    G, (n, k) = len(np.unique(groups)), X.shape
    V = G / (G - 1) * (n - 1) / max(n - k, 1) * XtX_inv @ meat @ XtX_inv
    return b, np.sqrt(np.maximum(np.diag(V), 0)), G


def cce(g, xs):
    """CCEP and CCEMG. The common factors are proxied by the cross-section averages (by sector and year) of y, k and of every stock.
    CCEP: every unit is purged of a constant and of these averages, then the residuals are pooled. CCEMG: one regression per unit, 5 % trimmed mean.
    Returns None if there are too few countries or units."""
    vars_ = ["y", "k"] + xs
    g = g[g.groupby("unit")["annee"].transform("size") >= MIN_T_CCE].copy()
    if g["Region"].nunique() < 5 or g["unit"].nunique() < 5:
        return None
    avg = g.groupby(["Sector", "annee"])[vars_].transform("mean")
    bars = []
    for v in vars_:
        g[f"bar_{v}"] = avg[v]
        bars.append(f"bar_{v}")
    parts, mg = [], []
    for u, gu in g.groupby("unit"):
        H = np.column_stack([np.ones(len(gu))] + [gu[b].to_numpy() for b in bars])
        P = np.linalg.pinv(H)
        R = {v: gu[v].to_numpy() - H @ (P @ gu[v].to_numpy()) for v in vars_}
        parts.append(pd.DataFrame({**R, "Sector": gu["Sector"].to_numpy(), "Region": gu["Region"].to_numpy()}))
        X = np.column_stack([np.ones(len(gu)), gu["k"].to_numpy()] + [gu[x].to_numpy() for x in xs] +
                            [gu[b].to_numpy() for b in bars])
        if len(gu) >= X.shape[1] + MIN_DOF_MG:
            bu = np.linalg.lstsq(X, gu["y"].to_numpy(), rcond=None)[0]
            mg.append(bu[2:2 + len(xs)])
    R = pd.concat(parts, ignore_index=True)
    secs = sorted(R["Sector"].unique())
    kcols = [np.where(R["Sector"] == s, R["k"], 0.0) for s in secs] if len(secs) > 1 else [R["k"].to_numpy()]
    X = np.column_stack([R[x].to_numpy() for x in xs] + kcols)
    b, se, G = cluster_ols(R["y"].to_numpy(), X, R["Region"].to_numpy())
    out = {}
    mg = np.array(mg) if mg else np.empty((0, len(xs)))
    for j, x in enumerate(xs):
        p = 2 * (1 - stats.t.cdf(abs(b[j] / se[j]), G - 1)) if se[j] > 0 else np.nan
        res = {"CCEP": (b[j], se[j], p)}
        c = mg[:, j][np.isfinite(mg[:, j])] if len(mg) else np.array([])
        if len(c) >= 5:
            lo, hi = np.quantile(c, [0.05, 0.95])
            ct = c[(c >= lo) & (c <= hi)]
            s = ct.std(ddof=1) / np.sqrt(len(ct))
            res["CCEMG"] = (ct.mean(), s, 2 * (1 - stats.norm.cdf(abs(ct.mean() / s))))
        else:
            res["CCEMG"] = (np.nan, np.nan, np.nan)
        out[x] = res
    return out, len(g), g["unit"].nunique(), len(mg)

In [ ]:
rows, failures = [], []
NOTED = set()


def run_sample(label, base, metric, d, sectors, sector_name, with_cce=True):
    """Estimates every formulation on one sample (one sector or one group of sectors) and appends the estimates to `rows`."""
    single = cross_section(base)
    for form, xs in FORMULATIONS.items():
        g = sample(d, sectors, SAMPLE_VARS.get(form, xs), min_obs=1 if single else None)
        if g["Region"].nunique() < 3 or len(g) < 30:
            continue
        common = dict(sample=label, sector=sector_name, base=base, metric=metric, formulation=form)
        if single:   # one year: only the cross-section estimation is possible (reported under "Country FE")
            try:
                r, n, u, eff = fe_cross(g, xs)
            except Exception as e:
                failures.append((base, metric, sector_name, form, "cross-section", repr(e)))
                continue
            for x, (b, s, p) in r.items():
                rows.append({**common, "variable": x, "method": "Country FE", "coef": b, "se": s, "pvalue": p, "nobs": n, "units": u, "effects": eff})
            if CCE_SECTOR_DIM:     # CCE with the countries as units and the sectors as periods
                try:
                    out = cce_sector(g, xs)
                except Exception as e:
                    failures.append((base, metric, sector_name, form, "CCE sector dim.", repr(e)))
                    out = None
                if isinstance(out, str):
                    if (base, metric) not in NOTED:
                        NOTED.add((base, metric))
                        print(f"[{base} {metric}] CCE over sectors skipped: a stock is the same for all the sectors of a country")
                elif out is not None:
                    r2, n2, u2, n_mg = out
                    for x, dct in r2.items():
                        for meth in ("CCEP", "CCEMG"):
                            b, s, p = dct[meth]
                            rows.append({**common, "variable": x, "method": meth, "coef": b, "se": s, "pvalue": p, "nobs": n2,
                                         "units": u2 if meth.startswith("CCEP") else n_mg, "effects": "countries = units, sectors = periods"})
            continue
        for meth, yfe in [("Country FE", False), ("Country FE + year FE", True)]:
            try:
                r, n, u = fe(g, xs, yfe)
            except Exception as e:
                failures.append((base, metric, sector_name, form, meth, repr(e)))
                continue
            for x, (b, s, p) in r.items():
                rows.append({**common, "variable": x, "method": meth, "coef": b, "se": s, "pvalue": p, "nobs": n, "units": u})
        if with_cce:
            try:
                out = cce(g, xs)
            except Exception as e:
                failures.append((base, metric, sector_name, form, "CCE", repr(e)))
                out = None
            if out is not None:
                r, n, u, n_mg = out
                for x, dct in r.items():
                    for meth in ("CCEP", "CCEMG"):
                        b, s, p = dct[meth]
                        rows.append({**common, "variable": x, "method": meth, "coef": b, "se": s, "pvalue": p,
                                     "nobs": n, "units": u if meth == "CCEP" else n_mg})


RUN_LIST = [(b, m, v) for b, m in RUNS for v in (FDI_VARIANTS if b == "FDI" else [None])]   # FDI: one run per measure
for base, metric, variant in RUN_LIST:
    label = f"{metric}_{variant}" if variant else metric          # metric name of the estimates and of the output file
    if not variant and not os.path.exists(panel_file(base)):
        print(f"[{base}] panel not found: {panel_file(base)}: skipped")
        continue
    try:
        d = prepare(base, metric, variant)
    except FileNotFoundError as e:
        print(f"[{base}] {e}: skipped")
        continue
    first_row = len(rows)
    no_cce = base in NO_CCE_BASES or cross_section(base)
    if not no_cce and not ONLY_POOLED:
        for sector in sorted(d["Sector"].dropna().unique()):
            run_sample("by_sector", base, label, d, [sector], sector)
    for gname, secs in GROUPS.items():
        run_sample("pooled", base, label, d, secs, gname, with_cce=not no_cce)
    if len(rows) == first_row:
        print(f"[{base} {label}] No estimates available for export.")
    else:
        print(f"[{base} {label}] {len(rows) - first_row} estimates", flush=True)

res = pd.DataFrame(rows)
if res.empty:
    raise RuntimeError(f"no estimate. RUNS = {RUNS}; panels read in {RESULTS_DIR}regression_panels/ (RESULTS_DIR exists: {os.path.isdir(RESULTS_DIR)}). "
                       "Check PATH_WRK in project_settings.py (the runs folder of your machine) and RUNS.")
res["cell"] = [("" if not np.isfinite(b) else f"{b:.3f}{stars(p)}") for b, p in zip(res["coef"], res["pvalue"])]
print(f"{len(res)} estimates" + (f", {len(failures)} estimations failed (see `failures`)" if failures else ""))

In [ ]:
form_order = list(FORMULATIONS)
var_order = list(dict.fromkeys(v for xs in FORMULATIONS.values() for v in xs))


def wide(sub, index):
    """One row per (index), one column per method, cells = "coef stars"; observations and units of the Country FE estimation."""
    t = sub.pivot_table(index=index, columns="method", values="cell", aggfunc="first")
    t = t.reindex(columns=[m for m in METHODS if m in t.columns])
    n = sub[sub.method == "Country FE"].groupby(index)[["nobs", "units"]].first()
    t = t.join(n.rename(columns={"nobs": "obs (FE)", "units": "units (FE)"}))
    t = t.reset_index()
    t["formulation"] = pd.Categorical(t["formulation"], form_order, ordered=True)
    t["variable"] = pd.Categorical(t["variable"], var_order, ordered=True)
    return t.sort_values([c for c in index if c in ("sector",)] + ["formulation", "variable", "base", "metric"])


def readme_rows():
    from datetime import datetime
    return [
        "Estimation of the unit-cost equation  ln(c_u / w_l) = alpha * ln(C_k / w_l) + gamma' X + e,  X = ln knowledge stocks.",
        "Methods: " + "; ".join(METHODS) + ". Country FE: country fixed effects (country x sector in the pooled estimations) + 2009 dummy; "
        "Country FE + year FE: with year fixed effects" + (" specific to each sector" if YEAR_FE_BY_SECTOR else " common to all the sectors of the group (pooled groups)") + "; "
        "CCEP / CCEMG: Common Correlated Effects (Pesaran 2006), pooled / mean of the units, 5 % trimmed.",
        "For the bases estimated on ONE year (FDI inward) CCEP / CCEMG are computed with the countries as units and their sectors as periods: the cross-section averages "
        "are taken by sector over the countries (minimum " + f"{MIN_SECTORS_CCE} sectors per country, {MIN_COUNTRIES_SECTOR} countries per sector, {MIN_DOF_MG} degrees of freedom for the CCEMG). "
        "Same coefficient for all the sectors of a country; needs a stock that varies across the sectors of a country.",
        "Standard errors: clustered by country (Country FE, year FE, CCEP); CCEMG: standard deviation of the units / sqrt(number of units).",
        "Formulations: " + "; ".join(f"{f} = {', '.join(xs)}" for f, xs in FORMULATIONS.items()) + ".",
        "Samples: by_sector (units = countries); pooled groups (units = country x sector, alpha specific to each sector): "
        + "; ".join(f"{g} = {len(s)} sectors" for g, s in GROUPS.items()) + ". The aggregates C and H are left out.",
        f"Bases and metrics: {RUNS} (FDI: measures {FDI_VARIANTS}); bases without CCE (pooled groups, two fixed-effects methods only): {NO_CCE_BASES}.",
        f"Units with fewer than {MIN_OBS_UNIT} observations are dropped; CCE: at least {MIN_T_CCE} observations per unit, 5 countries and 5 units; CCEMG: unit regressions with at least {MIN_DOF_MG} degrees of freedom.",
        *([f"Bases estimated on a single year (ONLY_YEAR = {ONLY_YEAR}): cross-section, pooled groups only; the column Country FE then holds an OLS with country effects, "
           "sector intercepts and alpha specific to each sector (no unit effects, no year effects, no CCE: they need a time dimension). "
           "Column 'effects' of the sheet raw: when a stock is the same for all the sectors of a country (FDI outward), it cannot be separated from the country effects: "
           "only the sector intercepts are kept ('sector')."] if any(b in ONLY_YEAR for b, _ in RUNS) else []),
        "",
        "Cells: coefficient with significance stars (*** 1 %, ** 5 %, * 10 %).",
        "Sign: a NEGATIVE coefficient means that the knowledge stock LOWERS the unit cost, i.e. RAISES TFP.",
        "Sheet 'synthesis': all metrics (rows) for the pooled groups and the methods (columns).",
        "Empty CCE cells: too few countries (< 5) or too short series; CCEMG also needs enough observations per unit for the unit regressions.",
        "Sheet 'raw': coefficient, standard error, p-value, observations and units for every estimate.",
        f"Generated {datetime.now():%Y-%m-%d %H:%M}.",
    ]


def export_cce(res, xlsx_path):
    """Excel file: README, synthesis, sample_sizes, one sheet per pooled group, by_sector, raw; csv of all the estimates next to it."""
    res.to_csv(xlsx_path.replace(".xlsx", ".csv"), index=False)
    pool = res[res["sample"] == "pooled"].copy()
    with pd.ExcelWriter(xlsx_path) as w:
        pd.DataFrame({"README": readme_rows()}).to_excel(w, sheet_name="README", index=False)
        syn = pool.pivot_table(index=["formulation", "variable", "base", "metric"], columns=["sector", "method"], values="cell", aggfunc="first")
        syn = syn.reindex(columns=pd.MultiIndex.from_product([list(GROUPS), [m for m in METHODS if m in set(res["method"])]])).reset_index()
        syn["formulation"] = pd.Categorical(syn["formulation"], form_order, ordered=True)
        syn["variable"] = pd.Categorical(syn["variable"], var_order, ordered=True)
        syn = syn.sort_values(["formulation", "variable", "base", "metric"]).set_index(["formulation", "variable", "base", "metric"])
        syn.columns = [f"{g} | {m}" for g, m in syn.columns]
        syn = syn.reset_index()
        syn.to_excel(w, sheet_name="synthesis", index=False)
        obs = pool[pool["method"] == "Country FE"].pivot_table(index=["base", "metric"], columns="sector", values=["nobs", "units"], aggfunc="first")
        obs.to_excel(w, sheet_name="sample_sizes")
        for gname in GROUPS:
            wide(pool[pool["sector"] == gname], ["formulation", "variable", "base", "metric"]).to_excel(w, sheet_name=f"pooled_{gname}", index=False)
        if (res["sample"] == "by_sector").any():
            wide(res[res["sample"] == "by_sector"], ["sector", "formulation", "variable", "base", "metric"]).to_excel(w, sheet_name="by_sector", index=False)
        res.drop(columns="cell").to_excel(w, sheet_name="raw", index=False)
    print(f"written {xlsx_path} ({len(res)} estimates)")
    return syn


syntheses = []
for (_base, _metric), sub in res.groupby(["base", "metric"], sort=False):
    out_path = RESULTS_DIR + f"estimation_results/{_base}/tab_{_base}_{_metric}{CHOC_TAG}{OUT_TAG}.xlsx"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    syntheses.append(export_cce(sub.copy(), out_path))
synthesis = pd.concat(syntheses, ignore_index=True)
synthesis
